# Commercial (shop) rent and site cost, all emirates

**Question:** what does a lounge site cost in each emirate, how good is each rent source, and how does each lounge's or area's rent compare with the salon sites in its emirate (`rent_ratio`)?

Since 2026-10-11 every cell has a shop rent (`src/data_v3.py`, `SHOP_RENT_WHY`), from one of three sources or its emirate's median. The JLL mall-class bands (`retail_rent_bands.csv`, `lounge_site_class.csv`) are no longer used.

**How the model works** is described once, in [docs/decision-model.md](../docs/decision-model.md#how-a-call-is-made-step-by-step) ("How a call is made, step by step"): demand from the Huff split (drive time stretches the decay, competitor coverage scales the competitors' pull), site rent, the within-emirate scorecard and the growth what-if. This notebook doesn't repeat it, so it can't drift from the code.

**Every parameter here is a first cut**, not calibrated. This notebook shows what they produce so we can decide together.

In [1]:
import sys, pathlib, statistics
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 120)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
from src.features import lounges as L
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
F = pd.DataFrame([x.model_dump() for x in r.features]).set_index("branch_id")
D = {d.branch_id: d for d in r.decisions}
AR = pd.DataFrame([x.model_dump(exclude={"cell_ids"}) for x in r.areas]).set_index("area_id")
AR["call"] = AR.index.map({d.area_id: d.action for d in r.area_decisions})

## 1. Sources

In [2]:
from src.data_v3 import SHOP_RENT_WHY
print(SHOP_RENT_WHY, "\n")
pd.DataFrame([
    dict(source="dld", emirates="Dubai", what="DLD Ejari shop contracts, median AED/m2 per DLD area, registered 2026-07-10 to 2026-10-09", kind="signed contracts", file="cell_shop_rent.csv"),
    dict(source="adrec", emirates="Abu Dhabi", what="ADREC 2025 street retail & community mall rate per district, read off the public map", kind="contracts (2025 average)", file="adrec_retail_rent_2025.csv"),
    dict(source="listings", emirates="Sharjah, Ajman, RAK, UAQ, Fujairah", what="Property Finder + Bayut shop-for-rent listings, median of the 10 nearest within 5 km (5+ needed)", kind="asking rents (2026-10-11)", file="cell_listing_rent.csv"),
    dict(source="emirate_median", emirates="all", what="median over the emirate's cells that have one of the above", kind="fallback", file="(computed)")]).set_index("source")

Shop rent (AED/m2/yr, our site cost) per cell, one column for every emirate, 2026-10-11. Dubai: DLD Ejari shop contracts (cell_shop_rent.csv). Abu Dhabi: ADREC 2025 street-retail contract rates by district (adrec_retail_rent_2025.csv), matched on the cell's OSM name. Northern emirates: median asking rent of nearby shop listings (cell_listing_rent.csv; asking, so above achieved rents). A cell with none of these (or a listings cell with under 5 listings near it) takes its emirate's median over the cells that have one: source emirate_median. 



,emirates,what,kind,file
source,,,,
dld,Dubai,"DLD Ejari shop contracts, median AED/m2 per DL...",signed contracts,cell_shop_rent.csv
adrec,Abu Dhabi,ADREC 2025 street retail & community mall rate...,contracts (2025 average),adrec_retail_rent_2025.csv
listings,"Sharjah, Ajman, RAK, UAQ, Fujairah",Property Finder + Bayut shop-for-rent listings...,asking rents (2026-10-11),cell_listing_rent.csv
emirate_median,all,median over the emirate's cells that have one ...,fallback,(computed)


## 2. Coverage by source, per emirate (cells and women)

In [3]:
from src.market import women_15plus
c = v3.cells.assign(women=women_15plus(v3.cells, v3.emirates, 0.055))
cov = pd.crosstab(c.emirate, c.shop_rent_source, margins=True)
wcov = (c.pivot_table(index="emirate", columns="shop_rent_source", values="women", aggfunc="sum", fill_value=0)
        .pipe(lambda t: t.div(t.sum(axis=1), axis=0) * 100).round(1))
display(cov); wcov.add_suffix(" % women")

shop_rent_source,adrec,dld,emirate_median,listings,All
emirate,,,,,
Abu Dhabi,760,0,407,0,1167
Ajman,0,0,19,33,52
Dubai,0,243,214,0,457
Fujairah,0,0,112,19,131
Ras Al Khaimah,0,0,133,62,195
Sharjah,0,0,246,82,328
Umm Al Quwain,0,0,20,23,43
All,760,243,1151,219,2373


shop_rent_source,adrec % women,dld % women,emirate_median % women,listings % women
emirate,,,,
Abu Dhabi,79.1,0.0,20.9,0.0
Ajman,0.0,0.0,17.7,82.3
Dubai,0.0,82.1,17.9,0.0
Fujairah,0.0,0.0,77.8,22.2
Ras Al Khaimah,0.0,0.0,45.9,54.1
Sharjah,0.0,0.0,50.9,49.1
Umm Al Quwain,0.0,0.0,32.3,67.7


## 3. Per-emirate distributions (AED/m²/yr)

Direct sources only (the emirate-median fallback cells would pile up on one value).

In [4]:
direct = c[c.shop_rent_source != "emirate_median"]
d = direct.groupby("emirate").shop_rent_m2.describe(percentiles=[.1, .25, .5, .75, .9]).round(0)
d["source"] = direct.groupby("emirate").shop_rent_source.first()
d["emirate_median_fallback"] = c[c.shop_rent_source == "emirate_median"].groupby("emirate").shop_rent_m2.first().round(0)
d

,count,mean,std,min,10%,25%,50%,75%,90%,max,source,emirate_median_fallback
emirate,,,,,,,,,,,,
Abu Dhabi,760.0,1212.0,743.0,165.0,289.0,646.0,1168.0,1658.0,2312.0,4537.0,adrec,1168.0
Ajman,33.0,814.0,187.0,502.0,592.0,674.0,755.0,972.0,1027.0,1226.0,listings,755.0
Dubai,243.0,1993.0,946.0,93.0,889.0,1364.0,1985.0,2441.0,3095.0,5774.0,dld,1985.0
Fujairah,19.0,540.0,126.0,461.0,461.0,461.0,461.0,623.0,745.0,745.0,listings,461.0
Ras Al Khaimah,62.0,798.0,367.0,357.0,380.0,447.0,819.0,1133.0,1291.0,1557.0,listings,819.0
Sharjah,82.0,1272.0,472.0,323.0,761.0,1046.0,1121.0,1704.0,1745.0,2354.0,listings,1121.0
Umm Al Quwain,23.0,668.0,192.0,397.0,419.0,543.0,665.0,846.0,846.0,1176.0,listings,665.0


## 4. Abu Dhabi: matching cells to ADREC districts

A cell takes its district's rate when its OSM name matches an ADREC district after normalising (ASCII, lowercase, no punctuation, no "Al", known spellings such as Ghiyathi = Ghayathi). Al Bateen and Al Rawdah exist in both Abu Dhabi City and Al Ain: told apart by longitude.

In [5]:
ad = c[c.emirate == "Abu Dhabi"]; m = ad.shop_rent_source == "adrec"
print(f"cells matched: {m.sum()} of {len(ad)} ({m.mean():.0%}) | women in matched cells: {ad.women[m].sum() / ad.women.sum():.0%}")
adrec = pd.read_csv(REPO_ROOT / "data/seed/v3/adrec_retail_rent_2025.csv")
from src.data_v3 import _norm
used = set(ad.name[m].map(_norm))
print("ADREC districts:", len(adrec), "| matched to a cell:", int(adrec.district.map(_norm).isin(used).sum()))
print("low-confidence rows (no 2025 street-retail contracts):", list(adrec.district[adrec.source_note.str.contains("low confidence")]))
print("Sabkhat Matti:", int(adrec.set_index("district").rent_aed_sqm["Sabkhat Matti"]), "AED/m2, the highest ADREC rate (Al Dhafra, kept as-is);",
      "its cells:", int((ad.name == "Sabkhat Matti").sum()))
print("\nunmatched names holding the most women:")
ad[~m].groupby("name").women.sum().sort_values(ascending=False).head(10).round(0)

cells matched: 760 of 1167 (65%) | women in matched cells: 79%
ADREC districts: 111 | matched to a cell: 101
low-confidence rows (no 2025 street-retail contracts): ['Al Marfa', 'Delma']
Sabkhat Matti: 4537 AED/m2, the highest ADREC rate (Al Dhafra, kept as-is); its cells: 1

unmatched names holding the most women:


name
Zayed Military City          33762.0
Al Ruways Industrial City    18309.0
Al Sad                        6698.0
Masakin                       6299.0
Al Nabbagh                    5363.0
Al Muzoun                     5295.0
Al Markhaniyyah               4984.0
Qasr Al Bahr                  4707.0
Muwaylih                      4667.0
Khalifa Industrial            4666.0
Name: women, dtype: float64

## 5. Northern emirates: listings after the two exclusions

Agreed with the owner (`scripts/build_listing_rent.py`): drop listings in industrial areas (location path matches the growth model's `NON_RESIDENTIAL`), and count at most 3 listings per building (pins within 30 m).

In [6]:
ls = pd.read_csv(REPO_ROOT / "data/seed/v3/listing_rents_clean.csv")
ls["excluded"] = ls.excluded.fillna("counted")
t = pd.crosstab(ls.emirate, ls.excluded, margins=True)
t["counted_median_aed_m2"] = ls[ls.excluded == "counted"].groupby("emirate").aed_per_m2.median().round(0)
t

excluded,building_cap,counted,industrial,All,counted_median_aed_m2
emirate,,,,,
Ajman,175,204,127,506,807.0
Fujairah,2,12,0,14,500.0
Ras Al Khaimah,21,96,0,117,682.0
Sharjah,83,259,93,435,1156.0
Umm Al Quwain,89,42,3,134,599.0
All,370,613,223,1206,NaN


## 6. Does shop rent track the salon market? Per emirate

Spearman, cells with a direct rent (no fallback). Salon level: each non-Bedashing salon's reviews against its cell's rent. Cell level: a cell's total and premium salon reviews against its rent. Shown where n ≥ 10.

In [7]:
from src.market import premium_substitutes, _reviews
sal = v3.salons[(v3.salons.excluded_reason == "") & ~v3.salons.is_bedashing.astype(bool)].reset_index()
lat, lng = sal.lat.to_numpy(float)[:, None], sal.lng.to_numpy(float)[:, None]
inbox = (c.south.to_numpy() <= lat) & (lat < c.north.to_numpy()) & (c.west.to_numpy() <= lng) & (lng < c.east.to_numpy())
sal["cell_id"] = np.where(inbox.any(1), c.index.to_numpy()[inbox.argmax(1)], None)
sal["reviews"] = [_reviews(x) for x in sal.to_dict("records")]
prem_ids = {x["place_id"] for x in premium_substitutes(sal.to_dict("records"), len(sal), a.premium_min_rating, a.comparable_price_levels)}
sal["premium"] = sal.place_id.isin(prem_ids)
sal = sal[sal.cell_id.isin(direct.index)]
sal["shop_rent"], sal["emirate"] = sal.cell_id.map(c.shop_rent_m2), sal.cell_id.map(c.emirate)
cell = pd.DataFrame({"shop": direct.shop_rent_m2, "emirate": direct.emirate, "res": direct.rent_observed,
                     "reviews": sal.groupby("cell_id").reviews.sum(), "prem_reviews": sal[sal.premium].groupby("cell_id").reviews.sum()}
                    ).loc[direct.index].fillna({"reviews": 0, "prem_reviews": 0})
cell = cell[cell.reviews > 0]   # cells with at least one salon
rows = []
def rho(x, y):
    return round(x.corr(y, method="spearman"), 2) if len(x) >= 10 else None
for em in sorted(c.emirate.unique()):
    s_, k = sal[sal.emirate == em], cell[cell.emirate == em]
    rows.append(dict(emirate=em, source=direct[direct.emirate == em].shop_rent_source.iloc[0] if (direct.emirate == em).any() else "-",
                     salons=len(s_), salon_reviews=rho(s_.reviews, s_.shop_rent),
                     premium_salons=int(s_.premium.sum()), premium_reviews=rho(s_[s_.premium].reviews, s_[s_.premium].shop_rent),
                     cells_with_salons=len(k), cell_reviews=rho(k.reviews, k.shop), cell_premium_reviews=rho(k.prem_reviews, k.shop)))
pd.DataFrame(rows).set_index("emirate")

,source,salons,salon_reviews,premium_salons,premium_reviews,cells_with_salons,cell_reviews,cell_premium_reviews
emirate,,,,,,,,
Abu Dhabi,adrec,1108,0.05,287,0.01,183,0.09,0.05
Ajman,listings,219,0.08,38,-0.06,29,0.07,0.15
Dubai,dld,1177,0.08,571,0.13,161,0.15,0.15
Fujairah,listings,94,0.31,17,-0.13,11,0.66,0.47
Ras Al Khaimah,listings,235,0.19,31,-0.28,39,0.14,0.25
Sharjah,listings,448,0.05,130,0.13,51,-0.06,-0.11
Umm Al Quwain,listings,71,0.27,5,NaN,9,NaN,NaN


## 7. Site cost: formula and the fit-out assumption

site cost (AED 1k/yr) = (shop rent/m² × REF_UNIT_M2 + FITOUT_AED_M2 × REF_UNIT_M2 ÷ FITOUT_YEARS) ÷ 1000

In [8]:
print("REF_UNIT_M2", L.REF_UNIT_M2, "|", L.REF_UNIT_WHY)
print("FITOUT_AED_M2", L.FITOUT_AED_M2, "| FITOUT_YEARS", L.FITOUT_YEARS, "|", L.FITOUT_WHY)
q = direct.shop_rent_m2.quantile([.1, .25, .5, .75, .9])
pd.DataFrame({"shop_rent_m2": q, "site_cost_k": [L.site_cost_k(x) for x in q],
              "fitout_share_of_cost": [L.FITOUT_AED_M2 / L.FITOUT_YEARS / (x + L.FITOUT_AED_M2 / L.FITOUT_YEARS) for x in q]}).round(2)

REF_UNIT_M2 150 | Lounge floor areas aren't public: 150 m2 is an assumed reference unit. One size for every site, so it sets the level of women-per-rent, never the ranking.
FITOUT_AED_M2 3000 | FITOUT_YEARS 5 | Assumed, not sourced (2026-10-11): a salon fit-out of about AED 3,000/m² (AED 280/sqft), written off over 5 years, on the 150 m² reference unit: about AED 90k a year on top of rent.


,shop_rent_m2,site_cost_k,fitout_share_of_cost
0.10,380.5,147.07,0.61
0.25,682.0,192.30,0.47
0.50,1236.0,275.40,0.33
0.75,1774.7,356.20,0.25
0.90,2340.2,441.03,0.20


## 8. Per lounge: rent source, site cost, rent ratio

In [9]:
t = F[~F.not_scored][["emirate", "site_served", "site_rent_m2", "site_rent_source", "site_cost_k", "rent_ratio", "rent_pct_uae", "demand_ref", "demand_pct", "demand_pct_uae"]]
print("rent source:", t.site_rent_source.value_counts().to_dict())
t.sort_values(["emirate", "rent_ratio"]).round(2)

rent source: {'adrec': 12, 'dld': 5, 'listings': 4, 'emirate_median': 2}


,emirate,site_served,site_rent_m2,site_rent_source,site_cost_k,rent_ratio,rent_pct_uae,demand_ref,demand_pct,demand_pct_uae
branch_id,,,,,,,,,,
khalifa-city-a,Abu Dhabi,6144.21,2257.0,adrec,428.55,0.62,0.76,Abu Dhabi,0.73,0.78
baniyas,Abu Dhabi,13036.45,2003.0,adrec,390.45,0.70,0.62,Abu Dhabi,0.99,0.98
shakhbout-city,Abu Dhabi,8685.03,1815.0,adrec,362.25,0.77,0.56,Abu Dhabi,0.87,0.89
ministries-complex,Abu Dhabi,3919.20,1756.0,adrec,353.40,0.79,0.54,Abu Dhabi,0.50,0.65
shahama,Abu Dhabi,9865.31,1747.0,adrec,352.05,0.80,0.54,Abu Dhabi,0.93,0.92
noya-plaza,Abu Dhabi,4227.69,1707.0,adrec,346.05,0.82,0.52,Abu Dhabi,0.52,0.67
westyas,Abu Dhabi,3259.40,1707.0,adrec,346.05,0.82,0.52,Abu Dhabi,0.45,0.57
delma,Abu Dhabi,1894.78,1507.0,adrec,316.05,0.93,0.40,Abu Dhabi,0.38,0.25
khaleej-al-arabi,Abu Dhabi,1358.59,1380.0,adrec,297.00,1.01,0.34,Abu Dhabi,0.22,0.11


## 9. The salon-site reference: emirate median rents, rent ratios of lounges and areas

Every lounge and area is judged against the frozen reference of competitor salon sites in its own emirate (`SITE_REFERENCE_WHY`). `rent_ratio` = the emirate's median shop rent at those sites ÷ the site's (> 1 cheaper), scored on a frozen P10–P90 scale.

In [10]:
pd.set_option("display.max_colwidth", 200)
print(L.SITE_REFERENCE_WHY, "\n")
print(scorecard.RENT_RATIO_WHY, "\n")
ref = L.site_reference()
print("reference sites:", len(ref), "| MIN_REF_SITES", L.MIN_REF_SITES, "| RENT_RATIO_P10", scorecard.RENT_RATIO_P10, "| RENT_RATIO_P90", scorecard.RENT_RATIO_P90)
display(ref.groupby("emirate").agg(sites=("place_id", "size"), ref_served_p25=("ref_served", lambda x: x.quantile(.25)),
        ref_served_median=("ref_served", "median"), ref_served_p75=("ref_served", lambda x: x.quantile(.75)),
        rent_median=("shop_rent_m2", "median")).assign(demand_ranked=lambda x: np.where(x.sites >= L.MIN_REF_SITES, "own emirate", "UAE-wide"))
        .sort_values("sites", ascending=False).round(0))
rr = ref.shop_rent_m2.groupby(ref.emirate).transform("median") / ref.shop_rent_m2
print("reference rent_ratio quantiles:", rr.quantile([.1, .25, .5, .75, .9]).round(3).to_dict())
ac = AR[AR.new_site_served.notna()]
pd.DataFrame({"lounges": t.groupby("emirate").size(),
              "lounge_rent_ratio_median": t.groupby("emirate").rent_ratio.median().round(2),
              "evaluated_areas": ac.groupby("emirate").size(),
              "area_rent_ratio_median": ac.groupby("emirate").rent_ratio.median().round(2)}).fillna({"lounges": 0, "evaluated_areas": 0})

Labels are judged within each emirate, against its market of salon SITES, frozen: every real (non-imputed) premium competitor salon in the served model, Bedashing excluded (1,123 sites; data/seed/v3/site_reference.csv, built once 2026-10-11 by scripts/build_site_reference.py). Per site: the cell shop rent, and the women a typical lounge would serve there with all our lounges open. Never against our own lounges (Fujairah and Ras Al Khaimah have one, Sharjah two), and nothing we open or close moves it. Demand = the share of the emirate's reference sites a site out-serves; an emirate with under 50 reference sites (Ajman 38, Ras Al Khaimah 33, Fujairah 18, Umm Al Quwain 5) uses the UAE-wide set instead: a percentile among a few dozen sites moves several points per site. Rent ratio = the emirate's median shop rent at its reference sites ÷ the site's shop rent (its own emirate's median even under 50 sites: rents are an emirate-level market, and a UAE-wide median would call every northern sit

,sites,ref_served_p25,ref_served_median,ref_served_p75,rent_median,demand_ranked
emirate,,,,,,
Dubai,582,1815.0,2328.0,2926.0,2154.0,own emirate
Abu Dhabi,304,1448.0,3950.0,6366.0,1394.0,own emirate
Sharjah,143,3490.0,5905.0,8474.0,1121.0,own emirate
Ajman,38,9496.0,11100.0,12130.0,753.0,UAE-wide
Ras Al Khaimah,33,6668.0,7102.0,8676.0,853.0,UAE-wide
Fujairah,18,6940.0,7964.0,8225.0,745.0,UAE-wide
Umm Al Quwain,5,7226.0,10977.0,11424.0,543.0,UAE-wide


reference rent_ratio quantiles: {0.1: 0.618, 0.25: 0.834, 0.5: 1.0, 0.75: 1.148, 0.9: 1.429}


,lounges,lounge_rent_ratio_median,evaluated_areas,area_rent_ratio_median
emirate,,,,
Abu Dhabi,14.0,0.87,61.0,1.11
Ajman,0.0,NaN,10.0,1.00
Dubai,5.0,1.02,73.0,1.03
Fujairah,1.0,1.55,0.0,NaN
Ras Al Khaimah,1.0,0.73,5.0,1.04
Sharjah,2.0,0.72,11.0,1.00


In [11]:
print("lounges with a rent ratio:", int(t.rent_ratio.notna().sum()), "of", len(t), "| without:", list(t.index[t.rent_ratio.isna()]))
print("areas with a what-if:", len(ac), "| with a rent ratio:", int(ac.rent_ratio.notna().sum()))
ac.groupby("emirate").agg(areas=("rent_ratio", "size"), rent_ratio_median=("rent_ratio", "median"),
                          rent_pct_uae_median=("rent_pct_uae", "median"),
                          cheaper_than_median_site=("rent_ratio", lambda x: int((x >= 1).sum())),
                          GROW=("call", lambda x: int((x == "GROW").sum())), WATCH=("call", lambda x: int((x == "WATCH").sum()))).round(2)

lounges with a rent ratio: 23 of 23 | without: []
areas with a what-if: 160 | with a rent ratio: 160


,areas,rent_ratio_median,rent_pct_uae_median,cheaper_than_median_site,GROW,WATCH
emirate,,,,,,
Abu Dhabi,61,1.11,0.30,37,13,35
Ajman,10,1.00,0.07,5,6,4
Dubai,73,1.03,0.65,41,1,22
Ras Al Khaimah,5,1.04,0.07,4,4,1
Sharjah,11,1.00,0.20,10,10,1


## Open questions for review

1. **Direct rent covers most women, but unevenly.** A source of its own (not the emirate median) covers 82% of the women in Dubai (DLD), 79% in Abu Dhabi (ADREC), and through listings 82% in Ajman, 68% in Umm Al Quwain, 54% in Ras Al Khaimah, 49% in Sharjah and 22% in Fujairah. 21 of 23 scored lounges sit in a cell with a direct rent; al-ain and al-falah take the Abu Dhabi median (AED 1,168/m², rent ratio 1.19), so their rent scores neutral (0.5) and their calls are low confidence (question 7).
2. **ADREC matching by name:** 760 of 1,167 Abu Dhabi cells (65%, 79% of the women) and 101 of 111 districts. The biggest misses are Zayed Military City (33,762 women; al-falah's site cell) and Al Ruways Industrial City. Delma and Al Marfa (no 2025 street-retail contracts) are kept, and so is Sabkhat Matti at AED 4,537/m², the highest rate, on one cell. Worth a district polygon join instead of matching names?
3. **Listings are asking rents, and thin after the exclusions.** 593 of 1,206 listings are excluded (223 industrial, 370 over the 3-per-building cap), leaving 613; Fujairah has 12 counted listings and 19 direct cells, Umm Al Quwain 42. Asking rents sit above achieved ones, so northern sites look dearer than contract rents would make them. That matters more now: al-jada's listing rent (ratio 0.63) is what makes it SHRINK (`scorecard.ipynb`).
4. **Shop rent barely tracks the salon market in any emirate** (section 6): salon-level Spearman is 0.05–0.31 for all salons and −0.28 to 0.13 for premium ones; cell-level it is −0.11 to 0.25 everywhere except Fujairah (0.66 and 0.47, on 11 cells). Rent is a cost, not a demand proxy.
5. **The fit-out is unsourced, but it is no longer scored.** It is 20%–61% of `site_cost_k` (rent p90 to p10 over direct cells), and `site_cost_k` is now display only. The cost signal, `rent_ratio`, is rent alone.
6. **Each emirate sets its own rent bar** (section 9). The median shop rent at reference salon sites is AED 2,154/m² in Dubai, 1,394 in Abu Dhabi, 1,121 in Sharjah, 853 in Ras Al Khaimah, 753 in Ajman, 745 in Fujairah and 543 in Umm Al Quwain (Fujairah on 18 sites, Umm Al Quwain on 5). Over all 1,123 sites the rent ratio runs from 0.618 at P10 to 1.429 at P90, which is the frozen scoring scale. The median lounge ratio is 1.02 in Dubai, 0.87 in Abu Dhabi, 0.72 in Sharjah, 0.73 in Ras Al Khaimah and 1.55 in Fujairah. A ratio means "cheap for its emirate"; `rent_pct_uae` keeps the absolute view. Fujairah's al-taif-mall is the cheapest rent in the reference set (0.00) and Dubai's city-walk is dearer than 98% of it (0.98). Is that the right framing for a chain that can choose which emirate to grow in?
7. **On fallback rents the GROW rent test is noise.** An area whose cells take the emirate median gets ratio = that median ÷ the reference median. In Sharjah that is 1,121.1 ÷ 1,121 = 1.000, where all 10 Sharjah GROW areas sat. In Ras Al Khaimah it is 818.7 ÷ 853 = 1.04. In Ajman it is 755 ÷ 753 = 0.997, so manama-ajman (35,094 net new) and mudayrah-ajman (18,651) were WATCH on a 0.3% gap. The same effect puts the median area ratio at 1.00 in Sharjah and Ajman. **Answered 2026-10-11 (owner):** fallback rents count as unknown. An area with half or more of its rented women on the fallback passes the rent test as neutral, at low confidence; a lounge on a fallback site cell (al-ain, al-falah) scores rent 0.5 (`growth.RENT_FALLBACK_WHY`). manama-ajman and mudayrah-ajman are now GROW (low).